# 08b Promotion Vs Nonpromotion EDA Audit Patch

In [1]:

from pathlib import Path
from datetime import datetime
import subprocess, zipfile, os
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 120)
STEP='08b_promotion_vs_nonpromotion_eda_audit_patch_260513'; EXP='C:/Code/ott-churn-prediction'; RUN='run_20260514_022322'
CWD=Path.cwd().resolve(); norm=lambda s:s.replace('\\','/').rstrip('/')
actual=subprocess.check_output(['git','rev-parse','--show-toplevel'],cwd=CWD,text=True).strip(); ROOT=Path(actual).resolve(); PARK=ROOT/'park.ingyeom'; NB=PARK/'notebook'/STEP/f'{STEP}.ipynb'; OUT0=PARK/'reports'/'eda'/STEP; ZIPD=PARK/'zip'; ZIPP=ZIPD/f'{STEP}_review_package.zip'
if norm(actual)!=norm(EXP): print('STOP repo root mismatch',actual); raise SystemExit(1)
OUT=OUT0/(f"run_{datetime.now():%Y%m%d_%H%M%S}" if OUT0.exists() and any(OUT0.iterdir()) else '')
OUT.mkdir(parents=True,exist_ok=True); ZIPD.mkdir(parents=True,exist_ok=True); made=[]
def inside(p):
    try: Path(p).resolve().relative_to(PARK.resolve()); return True
    except Exception: return False
def W(df,n):
    q=OUT/n; df.to_csv(q,index=False,encoding='utf-8-sig'); made.append(q); return q
def R(p): return pd.read_csv(p)
def stat(p):
    p=Path(p)
    if not p.exists(): return None
    s=p.stat(); return (s.st_size,s.st_mtime_ns)
def yn(x): return 'yes' if bool(x) else 'no'
def pf(x): return 'PASS' if bool(x) else 'FAIL'
src=PARK/'data'/'(광일)Membership_v2_with_derived_features.csv'
a06=PARK/'reports'/'audits'/'06_common_preprocessing_and_final_cohort_260513'; a05=PARK/'reports'/'audits'/'05b_column_role_dictionary_patch_260513'; a07=PARK/'reports'/'audits'/'07_AARRR_feature_mapping_260513'
f06=[a06/'06_primary_main_cohort_index.csv',a06/'06_primary_main_cohort_conservative_features.csv',a06/'06_feature_policy_from_05b.csv',a06/'06_final_checks.csv']
f05=[a05/'05b_canonical_column_role_dictionary.csv',a05/'05b_conservative_safe_candidate_columns.csv',a05/'05b_review_required_columns.csv',a05/'05b_forbidden_drop_columns.csv']
f07=[a07/'07_AARRR_mapping_conservative_features.csv',a07/'07_AARRR_feature_mapping_all_columns.csv',a07/'07_final_checks.csv']
b08=PARK/'reports'/'eda'/'08_promotion_vs_nonpromotion_eda_260513'; s08=b08/RUN
n08=['08_preflight_input_validation.csv','08_cohort_consistency_check.csv','08_promotion_group_overview.csv','08_promotion_target_2x2_main_cohort.csv','08_promotion_repurchase_rate_difference.csv','08_raw_vs_main_promotion_target_comparison.csv','08_conservative_feature_distribution_by_promotion.csv','08_conservative_feature_promotion_difference_summary.csv','08_conservative_feature_distribution_by_promotion_target.csv','08_top_descriptive_differences_by_AARRR_stage.csv','08_AARRR_summary_by_promotion.csv','08_flag_distribution_by_promotion.csv','08_review_columns_excluded_from_standard_eda.csv','08_descriptive_findings_summary.csv','08_safe_unsafe_wording.csv','08_open_risks_for_next_steps.csv','08_final_checks.csv','README.md']
f08=[s08/x for x in n08]; src0=stat(src); st080={str(p):stat(p) for p in f08}; b08_before=list(b08.iterdir()) if b08.exists() else []
pre={'expected_repo_root':EXP,'actual_repo_root':actual,'repo_root_match':norm(actual)==norm(EXP),'source_file_exists':src.exists(),'required_06_files_exist':all(p.exists() for p in f06),'required_05b_files_exist':all(p.exists() for p in f05),'required_07_files_exist':all(p.exists() for p in f07),'required_08_success_run_folder_exists':s08.exists(),'all_required_08_success_run_files_exist':all(p.exists() for p in f08),'note_md_exists':(PARK/'note.md').exists(),'source_file_inside_park_ingyeom':inside(src),'output_folder_inside_park_ingyeom':inside(OUT),'notebook_inside_park_ingyeom':inside(NB),'zip_folder_inside_park_ingyeom':inside(ZIPD)}
pre['can_proceed']=all(v for k,v in pre.items() if k!='note_md_exists')
W(pd.DataFrame([pre]),'08b_preflight_input_validation.csv')
if not pre['can_proceed']:
    rp=OUT/'README.md'; rp.write_text('# 08b stopped\nRequired input missing. Only preflight and README were written.\n',encoding='utf-8'); made.append(rp); display(pd.DataFrame([pre])); raise SystemExit(1)
co=R(f06[1]); safe=R(f05[1]); rev=R(f05[2]); forb=R(f05[3]); map7=R(f07[0]); mapall=R(f07[1])
go=R(s08/'08_promotion_group_overview.csv'); t2=R(s08/'08_promotion_target_2x2_main_cohort.csv'); rd=R(s08/'08_promotion_repurchase_rate_difference.csv'); cc=R(s08/'08_cohort_consistency_check.csv'); fd=R(s08/'08_conservative_feature_distribution_by_promotion.csv'); diff=R(s08/'08_conservative_feature_promotion_difference_summary.csv'); fdt=R(s08/'08_conservative_feature_distribution_by_promotion_target.csv'); top=R(s08/'08_top_descriptive_differences_by_AARRR_stage.csv'); ar=R(s08/'08_AARRR_summary_by_promotion.csv'); rex=R(s08/'08_review_columns_excluded_from_standard_eda.csv')
meta={'source_row_number','USER_KEY','is_promotion','is_repurchase','duration_days'}; flags={c for c in co.columns if c.startswith('flag_')}; feats=[c for c in co.columns if c not in meta and c not in flags]
revcols=set(rev.column_name.astype(str)) if 'column_name' in rev else set(); forbcols=set(forb.column_name.astype(str)) if 'column_name' in forb else set(); safecols=set(safe.column_name.astype(str)) if 'column_name' in safe else set()
# B inventory and lock
inv=[]
for folder in [b08,s08]:
    for p in sorted(folder.iterdir(),key=lambda x:x.name.lower()):
        ok=(p==s08) or (s08 in p.parents); inv.append({'path':str(p.relative_to(PARK)),'item_type':'folder' if p.is_dir() else 'file','file_size_bytes':p.stat().st_size if p.is_file() else '', 'last_modified_time':datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec='seconds'),'belongs_to_success_run':yn(ok),'should_be_used_downstream':yn(ok),'reason':'Locked valid 08 source.' if ok else 'Ignored: base 08 folder may include failed or partial first-run artifacts.'})
W(pd.DataFrame(inv),'08b_08_run_folder_inventory.csv')
ignored=';'.join(str(p.relative_to(PARK)) for p in b08.iterdir() if p!=s08)
W(pd.DataFrame([{'valid_08_output_folder':str(s08.relative_to(PARK)),'invalid_or_ignored_08_locations':ignored,'reason_for_locking_success_run':'First 08 execution failed; successful run folder has required files and final checks.','first_run_failure_note':'Base folder artifacts are inventoried and ignored, not deleted.','downstream_requirement':'future steps must read 08 from run_20260514_022322'}]),'08b_08_source_of_truth_lock.csv')
# C recompute
non=co[co.is_promotion==0]; pro=co[co.is_promotion==1]; nr=non.is_repurchase.mean(); pr=pro.is_repurchase.mean(); od=pr-nr; pred=[c for c in co.columns if 'predict' in c.lower() or 'prediction' in c.lower()]
vals={'primary_main_cohort_row_count':len(co),'nonpromotion_row_count':len(non),'promotion_row_count':len(pro),'overall_repurchase_rate':co.is_repurchase.mean(),'nonpromotion_repurchase_rate':nr,'promotion_repurchase_rate':pr,'absolute_difference_promotion_minus_nonpromotion':od,'percentage_point_difference':od,'conservative_feature_count':len(feats),'review_columns_in_conservative_table_count':len(set(feats)&revcols),'forbidden_columns_in_conservative_table_count':len(set(feats)&forbcols),'has_repurchase_score_column':'repurchase_score' in co.columns,'has_churn_risk_column':'churn_risk' in co.columns,'has_prediction_column':len(pred)>0}
def g08(k):
    try:
        if k=='primary_main_cohort_row_count': return go.loc[go.is_promotion.astype(str).eq('overall'),'row_count'].iloc[0]
        if k=='nonpromotion_row_count': return go.loc[go.is_promotion.astype(str).eq('0'),'row_count'].iloc[0]
        if k=='promotion_row_count': return go.loc[go.is_promotion.astype(str).eq('1'),'row_count'].iloc[0]
        if k=='overall_repurchase_rate': return go.loc[go.is_promotion.astype(str).eq('overall'),'repurchase_rate'].iloc[0]
        if k in rd.columns: return rd[k].iloc[0]
        if k=='conservative_feature_count': return cc.loc[cc.check_name.eq('conservative actual feature count'),'actual_value'].iloc[0]
        if k=='review_columns_in_conservative_table_count': return cc.loc[cc.check_name.eq('no review columns appear as conservative feature columns'),'actual_value'].iloc[0]
        if k=='forbidden_columns_in_conservative_table_count': return cc.loc[cc.check_name.eq('no forbidden columns appear as conservative feature columns'),'actual_value'].iloc[0]
        if k in ['has_repurchase_score_column','has_churn_risk_column','has_prediction_column']: return False
    except Exception: return np.nan
    return np.nan
def mt(a,b):
    if pd.isna(b): return 'not_available'
    if isinstance(a,(bool,np.bool_)): return 'match' if bool(a)==bool(b) else 'mismatch'
    try: return 'match' if np.isclose(float(a),float(b),atol=1e-10) else 'mismatch'
    except Exception: return 'match' if str(a)==str(b) else 'mismatch'
mr=[]
for k,v in vals.items():
    b=g08(k); m=mt(v,b); mr.append({'metric_name':k,'recomputed_value':v,'value_from_08_if_available':b,'match_status':m,'interpretation':'consistent or not available' if m!='mismatch' else 'mismatch requires review'})
MR=pd.DataFrame(mr); W(MR,'08b_key_metric_recomputation.csv')
# D consistency
chk=[]
def C(n,ok,ev,act='No action.'): chk.append({'check_name':n,'status':'PASS' if ok else 'FAIL','evidence':ev,'required_action':act if ok else 'Review before downstream use.'})
cgo={int(r.is_promotion):int(r.row_count) for _,r in go[go.is_promotion.astype(str).isin(['0','1'])].iterrows()}; c2=t2.groupby('is_promotion')['count'].sum().to_dict(); cfd=fd.groupby('is_promotion')['n'].first().to_dict()
C('promotion row counts match across group overview, 2x2, feature distributions',cgo==c2 and all(cfd.get(k)==v for k,v in cgo.items()),f'overview={cgo};2x2={c2};dist={cfd}')
C('repurchase rates match across group overview and rate difference file',np.isclose(nr,rd.nonpromotion_repurchase_rate.iloc[0]) and np.isclose(pr,rd.promotion_repurchase_rate.iloc[0]),f'non={nr};pro={pr}')
C('feature count matches 22 after excluding metadata, target, split, flags, source_row_number, duration audit fields',len(feats)==22,';'.join(feats))
C('AARRR mapping does not treat __summary rows as features',not any(mapall.get('column_name',pd.Series(dtype=str)).astype(str).str.startswith('__summary') & mapall.get('column_name',pd.Series(dtype=str)).astype(str).isin(feats)),'__summary rows excluded')
pcols=[c for d in [diff,fdt,top] for c in d.columns if 'p_value' in c.lower() or c.lower() in ['p','pvalue']]; C('no p-value columns exist',len(pcols)==0,str(pcols)); C('no statistical test output exists',not any('test' in c.lower() or 'signif' in c.lower() for d in [diff,fdt,top] for c in d.columns),'checked columns')
model=[c for d in [diff,fdt,go] for c in d.columns if 'score' in c.lower() or 'predict' in c.lower()]; C('no model score columns exist',len(model)==0,str(model)); shap=[c for d in [diff,fdt,top] for c in d.columns if 'shap' in c.lower()]; C('no SHAP columns exist',len(shap)==0,str(shap))
fdnames=set(fd.feature_name.astype(str)); C('no review columns used in standard feature distribution',len(fdnames&revcols)==0,str(sorted(fdnames&revcols))); C('no forbidden columns used in standard feature distribution',len(fdnames&forbcols)==0,str(sorted(fdnames&forbcols)))
CHK=pd.DataFrame(chk); W(CHK,'08b_internal_consistency_audit.csv')
# E SMD audit
def buck(x):
    if pd.isna(x): return 'not_computable'
    x=abs(float(x)); return 'negligible' if x<.1 else 'small' if x<.2 else 'moderate' if x<.5 else 'large'
fa=diff.copy(); fa['standardized_mean_difference']=fa.simple_standardized_mean_difference; fa['absolute_standardized_mean_difference']=fa.standardized_mean_difference.abs(); fa['descriptive_effect_size_bucket_from_08']=fa.descriptive_effect_size_bucket; fa['recomputed_bucket_using_thresholds']=fa.standardized_mean_difference.apply(buck); fa['bucket_match_status']=np.where(fa.descriptive_effect_size_bucket_from_08.astype(str).str.lower().eq(fa.recomputed_bucket_using_thresholds),'match','mismatch'); fa['interpretation_strength']=fa.recomputed_bucket_using_thresholds.map({'large':'strong','moderate':'moderate','small':'weak','negligible':'negligible'}).fillna('not_computable'); fa['allowed_claim']='Descriptive effect size only; negligible values must not be overclaimed.'; fa['forbidden_claim']='Do not claim promotion caused this or that promotion rows have a strongly different profile.'; fa['next_analysis_need']='09 promotion x repurchase 2x2 within-group target comparison.'
W(fa[['feature_name','AARRR_stage_primary','feature_family','standardized_mean_difference','absolute_standardized_mean_difference','descriptive_effect_size_bucket_from_08','recomputed_bucket_using_thresholds','bucket_match_status','nonpromotion_mean','promotion_mean','mean_difference_promotion_minus_nonpromotion','interpretation_strength','allowed_claim','forbidden_claim','next_analysis_need']],'08b_conservative_feature_difference_interpretation_audit.csv')
bc=fa.recomputed_bucket_using_thresholds.value_counts().to_dict(); mx=float(fa.absolute_standardized_mean_difference.max())
W(pd.DataFrame([{'max_absolute_SMD_across_conservative_features':mx,'count_negligible_SMD':bc.get('negligible',0),'count_small_SMD':bc.get('small',0),'count_moderate_SMD':bc.get('moderate',0),'count_large_SMD':bc.get('large',0),'conclusion_about_promotion_vs_nonpromotion_average_feature_differences':'Conservative feature average differences are negligible overall; 08 does not support strong behavioral-profile separation by promotion alone.','safe_wording':'08에서는 재구매율 차이는 뚜렷하게 관찰되었지만, conservative feature 기준 평균 행동 차이는 전반적으로 negligible이었다.','unsafe_wording':'08에서 프로모션/비프로모션의 행동 차이가 뚜렷하게 드러났다.','why_this_is_not_a_failure':'A weak feature-difference result is a valid descriptive audit result.','why_09_is_needed':'09 should inspect promotion x repurchase 2x2 and within-group target differences.'}]),'08b_promotion_feature_difference_negative_finding.csv')
# F preview
prev=[]
for _,r in top.iterrows():
    comp=str(r.comparison_type)
    if 'within_promotion' not in comp and 'within_nonpromotion' not in comp: continue
    pg='promotion' if 'within_promotion' in comp else 'nonpromotion'
    for f,v in zip(str(r.top_features).split(';'),str(r.top_metric_values).split(';')):
        try: num=float(v)
        except Exception: num=np.nan
        prev.append({'feature_name':f,'AARRR_stage_primary':r.AARRR_stage_primary,'promotion_group':pg,'direction':'repurchase rows higher' if pd.notna(num) and num>0 else 'repurchase rows lower' if pd.notna(num) and num<0 else 'not computable','descriptive_strength':buck(num),'safe_preview_interpretation':'Limited preview from 08 only; do not treat as full 09.','why_09_must_analyze_deeper':'09 must compute promotion x repurchase cells directly.'})
W(pd.DataFrame(prev),'08b_promotion_target_signal_preview_from_08.csv')
# G AARRR restricted
arow=[]; repl=[]
for _,r in ar.iterrows():
    st=str(r.AARRR_stage); fs='' if pd.isna(r.feature_names) else str(r.feature_names); mix=st in ['activation','retention'] and ';' in fs; unsafe=mix or st in ['acquisition','revenue_proxy']
    arow.append({'AARRR_stage':st,'stage_level_feature_mean_averages_combine_heterogeneous_units_or_scales':yn(mix),'activation_and_retention_feature_lists_include_mixed_units':yn(st in ['activation','retention'] and mix),'stage_average_is_unsafe_to_interpret':yn(unsafe),'safe_to_use_parts':'feature_names; number_of_features only','parts_to_ignore':'raw stage mean averages','required_action':'Restrict raw stage averages from interpretation.'})
W(pd.DataFrame(arow),'08b_AARRR_summary_interpretability_audit.csv')
for st in ['acquisition','activation','retention','revenue_proxy','referral']:
    rr=ar[ar.AARRR_stage.astype(str).eq(st)]; fs=rr.feature_names.iloc[0] if len(rr) and pd.notna(rr.feature_names.iloc[0]) else ''; nf=int(rr.number_of_features.iloc[0]) if len(rr) else 0
    if st=='referral': obs='not_observed'; use='no'; safe_sum='Referral is not observed.'; unsafe='Do not claim referral performance was analyzed.'; reason='No referral variable.'
    elif st in ['activation','retention']: obs='proxy_features'; use='restricted'; safe_sum='Use feature list/count and individual feature distributions only.'; unsafe='Do not interpret heterogeneous raw stage mean averages.'; reason='Raw cross-feature averages are restricted.'
    elif st=='acquisition': obs='split_metadata'; use='restricted'; safe_sum='is_promotion is split metadata.'; unsafe='Do not interpret acquisition as ordinary feature average.'; reason='Split context only.'
    else: obs='target_proxy'; use='restricted'; safe_sum='Revenue proxy is target context only.'; unsafe='Do not treat as actual revenue feature.'; reason='Target proxy, not ordinary feature.'
    repl.append({'AARRR_stage':st,'number_of_conservative_features':nf,'feature_names':fs,'stage_is_directly_observed_or_proxy':obs,'safe_summary':safe_sum,'unsafe_summary':unsafe,'use_in_report':use,'reason':reason})
W(pd.DataFrame(repl),'08b_AARRR_summary_safe_replacement.csv')
# H validation
rows=[]; appeared=set(fd.feature_name.astype(str))
for col in sorted(safecols|revcols|forbcols|meta|{'is_promotion','is_repurchase'}):
    cat='target' if col=='is_repurchase' else 'split' if col=='is_promotion' else 'forbidden' if col in forbcols else 'review' if col in revcols else 'safe' if col in safecols else 'metadata'
    should=cat=='safe' and col in feats; did=col in appeared
    rows.append({'column_name':col,'category':cat,'appeared_in_08_feature_distribution':yn(did),'should_have_appeared':yn(should),'status':'PASS' if did==should else 'FAIL','note':'Review columns may appear in review-exclusion table only.' if cat=='review' else 'Standard conservative feature distribution rule checked.'})
W(pd.DataFrame(rows),'08b_review_column_exclusion_validation.csv')
# I to M fixed outputs
guard=[['repurchase_rate_difference','primary main cohort에서 프로모션 행의 재구매율은 비프로모션 행보다 낮게 관찰되었다.','이 차이는 descriptive이며, 프로모션의 인과효과를 의미하지 않는다.','primary main cohort에서 프로모션 행의 재구매율은 비프로모션 행보다 낮게 관찰되었다.','100원딜 때문에 재구매율이 낮아졌다.','use','09에서 promotion x repurchase 2x2 구조를 본다.'],['promotion_feature_average_difference','conservative safe feature 기준 promotion/non-promotion 평균 차이는 전반적으로 negligible 수준이었다.','따라서 08만으로 행동 구조가 크게 다르다고 주장하지 않는다.','conservative safe feature 기준 promotion/non-promotion 평균 차이는 전반적으로 negligible 수준이었다.','프로모션 고객은 행동 패턴이 확실히 다르다.','use as negative finding','09에서 집단 내부 target 차이를 본다.'],['AARRR_stage_summary','AARRR stage별 feature 목록과 count context만 사용할 수 있다.','08에서 AARRR 전체가 검증되었다는 주장은 지원하지 않는다.','08_AARRR_summary_by_promotion.csv raw stage mean average는 해석 제한한다.','08에서 AARRR 전체가 검증되었다.','restricted','개별 feature 분포를 본다.'],['promotion_target_internal_difference_preview','09에서 promotion x repurchase 2x2 구조를 더 깊게 볼 필요가 있다.','08b preview만으로 full 09 결론을 낼 수 없다.','09에서 promotion x repurchase 2x2 구조를 통해 각 집단 내부의 재구매/미재구매 차이를 더 깊게 본다.','08에서 충분히 설명됐다.','handoff only','09_promotion_repurchase_2x2_eda_260513'],['review_columns_excluded','Review columns were excluded from standard conservative feature comparison.','Review columns are approved standard features.','Review columns remain excluded.','보수 feature만으로 프로모션 집단의 행동 차이를 충분히 설명했다.','use','Keep excluded unless separately approved.'],['referral_boundary','Referral is not observed.','Referral 성과가 분석되었다.','Referral is not observed and remains future experiment proposal only.','Referral 성과가 분석되었다.','boundary','Keep out of 09.'],['causal_boundary','Descriptive row-level association only.','Causal effect of promotion.','이 차이는 descriptive이며, 프로모션의 인과효과를 의미하지 않는다.','프로모션 때문에 재구매율이 낮다.','mandatory caution','Maintain boundary.'],['next_step_09','Promotion split alone is insufficient for strong behavior-profile separation.','A final segmentation or model-ready conclusion.','09에서 promotion x repurchase 2x2 구조를 통해 각 집단 내부의 재구매/미재구매 차이를 더 깊게 본다.','08에서 충분히 설명됐다.','handoff','09_promotion_repurchase_2x2_eda_260513']]
W(pd.DataFrame(guard,columns=['topic','what_08_actually_supports','what_08_does_not_support','safe_claim','unsafe_claim','report_usage','next_step']),'08b_interpretation_guardrail.csv')
qs=[['Q1','Are promotion non-repurchase rows behaviorally different from promotion repurchase rows?','Within promotion rows, compare conservative features by is_repurchase.','06_primary_main_cohort_conservative_features.csv','is_promotion=1; is_repurchase','counts; rates; means; descriptive SMD','22 conservative safe features only','promotion_repurchase vs promotion_nonrepurchase table','Descriptive only.','No p-values, no causality.'],['Q2','Are non-promotion non-repurchase rows behaviorally different from non-promotion repurchase rows?','Within non-promotion rows, compare conservative features by is_repurchase.','06_primary_main_cohort_conservative_features.csv','is_promotion=0; is_repurchase','counts; rates; means; descriptive SMD','22 conservative safe features only','nonpromotion_repurchase vs nonpromotion_nonrepurchase table','Descriptive only.','No unique-user claim.'],['Q3','Are the features separating repurchase/non-repurchase similar or different across promotion groups?','Compare target-difference rankings across promotion groups.','06_primary_main_cohort_conservative_features.csv','is_promotion x is_repurchase','ranked descriptive differences','22 conservative safe features only','cross-group signal comparison','Descriptive comparison only.','No causality.'],['Q4','Are week3 usage features stronger within target comparison than promotion-level average comparison?','Compare week3 target-cell differences versus promotion averages.','06_primary_main_cohort_conservative_features.csv','is_promotion x is_repurchase','descriptive differences; buckets','week3 conservative usage features','week3-focused summary','Week3 may be stronger descriptively within target cells.','Day 21 onward forbidden.'],['Q5','Does the repurchase-rate gap remain descriptive only?','Document no causal claim.','08b_interpretation_guardrail.csv','is_promotion; is_repurchase','row counts; rates','target and split only','causal-boundary statement','Observed descriptively.','No causal language.'],['Q6','Which AARRR stage should 09 focus on based on conservative feature signals?','Use individual feature signals, not stage raw averages.','08b_AARRR_summary_safe_replacement.csv','is_promotion x is_repurchase','feature counts; individual differences','activation/retention conservative features','stage focus recommendation','Stage context through feature lists only.','Do not use raw stage means.']]
W(pd.DataFrame(qs,columns=['question_id','business_question','data_question','required_input_table','grouping_variable','metrics','feature_scope','expected_output','safe_interpretation','caution']),'08b_handoff_to_09_question_design.csv')
fails=int((CHK.status=='FAIL').sum()); mism=int((MR.match_status=='mismatch').sum()); accept=fails==0 and mism==0
W(pd.DataFrame([{'accept_08_structure':'yes' if accept else 'no','rerun_08_full':'no' if accept else 'yes','patch_interpretation':'yes','proceed_to_09':'yes' if accept else 'no','valid_08_run_folder':RUN,'whether_08_should_be_accepted':'yes' if accept else 'no','whether_08_should_be_rerun':'no' if accept else 'yes','whether_08_needs_full_replacement':'no' if accept else 'review_required','whether_08_needs_interpretation_guardrail':'yes','recommended_next_step':'09_promotion_repurchase_2x2_eda_260513','reason':'08 structure is consistent and key metrics match; interpretation guardrail is required.' if accept else 'Audit failures or metric mismatches require review.'}]),'08b_decision_summary.csv')
W(pd.DataFrame([['08에서 프로모션/비프로모션의 행동 차이가 뚜렷하게 드러났다.','08에서는 재구매율 차이는 뚜렷하게 관찰되었지만, conservative feature 기준 평균 행동 차이는 전반적으로 negligible이었다.'],['08 결과가 약하니 다른 방식으로 다시 돌려 강한 차이를 찾아야 한다.','08의 약한 feature 차이 자체를 결과로 인정하고, 09에서 promotion x repurchase 2x2 구조로 질문의 해상도를 높인다.'],['stage 평균값으로 Activation/Retention이 더 높다고 말할 수 있다.','서로 단위가 다른 feature의 stage 평균값은 해석하지 않고, stage별 feature 목록과 개별 feature 분포를 본다.'],['프로모션 때문에 재구매율이 낮다.','프로모션 행에서 낮은 재구매율이 관찰되었지만, 인과효과는 검증하지 않았다.'],['08에서 충분히 설명됐다.','08은 1차 promotion split 비교이며, 09와 10에서 target 내부 차이와 feature 분포를 더 봐야 한다.']],columns=['unsafe_wording','safer_wording']),'08b_safe_unsafe_wording.csv')
W(pd.DataFrame({'risk_to_carry_forward':['08 has descriptive repurchase-rate difference but weak promotion-level conservative feature differences.','09 must not overclaim; it should examine 2x2 structure.','10 must examine feature distributions more deeply.','Review columns remain excluded.','Membership/context L0 remains limited under conservative approach.','Content/genre review columns still unresolved.','Referral remains not observed.','Causal language remains forbidden.','Use 08 success run folder only.','Do not use AARRR stage raw mean averages from 08.']}),'08b_open_risks_for_next_steps.csv')
readme=OUT/'README.md'; readme.write_text(f'# 08b Promotion Vs Nonpromotion EDA Audit Patch\n\nThis is 08b audit/patch step only.\n08b does not rerun 08 with a new direction.\n08b locks the successful 08 run folder: reports/eda/08_promotion_vs_nonpromotion_eda_260513/{RUN}\n\nNo modeling was performed.\nNo predictions were created.\nNo repurchase_score or churn_risk was created.\nNo SHAP was performed.\nNo Optuna was performed.\nNo statistical significance testing was performed.\nNo p-values were created.\nNo feature engineering was performed.\nNo additional row exclusion was performed.\nReview columns were not used in standard conservative feature comparison.\n08 supports descriptive repurchase-rate difference by promotion.\n08 does not support strong promotion/non-promotion behavioral profile difference based on conservative feature averages.\n08 AARRR stage raw average values should not be used for interpretation.\n09 should examine promotion x repurchase 2x2 structure.\nNext recommended step is 09_promotion_repurchase_2x2_eda_260513.\n',encoding='utf-8'); made.append(readme)
note=PARK/'note.md'; old=note.read_text(encoding='utf-8') if note.exists() else '# Project note\n'; section=f"\n\n## {datetime.now():%Y-%m-%d %H:%M:%S} - {STEP}\n\n- purpose: 08 해석 위험 패치, 성공 08 run folder source lock, 09 handoff.\n- files created: {', '.join([p.name for p in made])}\n- valid 08 run folder: reports/eda/08_promotion_vs_nonpromotion_eda_260513/{RUN}\n- recomputed key metrics: primary={len(co):,}; nonpromotion={len(non):,}; promotion={len(pro):,}; nonpromotion repurchase rate={nr:.6f}; promotion repurchase rate={pr:.6f}; max abs SMD={mx:.6f}; SMD buckets={bc}\n- final interpretation of 08: 재구매율 차이는 descriptive하게 관찰되지만, conservative feature 평균 차이는 전반적으로 negligible이므로 행동 프로필이 크게 다르다고 주장하지 않는다.\n- restricted 08 outputs: 08_AARRR_summary_by_promotion.csv raw stage mean averages; base-folder 08 artifacts outside {RUN}; review columns as standard feature interpretation.\n- whether 08 should be rerun: {'no' if accept else 'yes'}\n- checks passed or failed: audit_fail_count={fails}; metric_mismatch_count={mism}; accept_08_structure={'yes' if accept else 'no'}\n- risks to carry forward: causal language forbidden; referral not observed; review columns excluded; AARRR raw stage averages restricted; 09 must not overclaim.\n- next step recommendation: 09_promotion_repurchase_2x2_eda_260513\n"; note.write_text(old.rstrip()+section,encoding='utf-8'); made.append(note)
# final checks then zip
csvs=['08b_preflight_input_validation.csv','08b_08_run_folder_inventory.csv','08b_08_source_of_truth_lock.csv','08b_key_metric_recomputation.csv','08b_internal_consistency_audit.csv','08b_conservative_feature_difference_interpretation_audit.csv','08b_promotion_feature_difference_negative_finding.csv','08b_promotion_target_signal_preview_from_08.csv','08b_AARRR_summary_interpretability_audit.csv','08b_AARRR_summary_safe_replacement.csv','08b_review_column_exclusion_validation.csv','08b_interpretation_guardrail.csv','08b_handoff_to_09_question_design.csv','08b_decision_summary.csv','08b_safe_unsafe_wording.csv','08b_open_risks_for_next_steps.csv','08b_final_checks.csv']
fc_names=['repo_root_checked','repo_root_matches_expected','source_file_exists','source_file_inside_park_ingyeom','required_06_files_exist','required_05b_files_exist','required_07_files_exist','required_08_success_run_folder_exists','all_required_08_success_files_exist','notebook_inside_park_ingyeom','output_folder_inside_park_ingyeom','zip_inside_park_ingyeom','no_files_written_outside_park_ingyeom','no_py_script_created','no_existing_notebook_modified','no_source_csv_modified','no_08_outputs_overwritten','no_files_deleted','no_modeling_performed','no_predictions_created','no_repurchase_score_created','no_churn_risk_created','no_shap_performed','no_optuna_performed','no_statistical_tests_performed','no_p_values_created','no_feature_engineering_performed','no_additional_rows_excluded','success_run_folder_locked','base_folder_partial_outputs_inventoried_if_present','key_metrics_recomputed','key_metrics_match_08_or_mismatches_recorded','conservative_feature_differences_audited','negligible_promotion_feature_difference_documented','AARRR_stage_average_interpretation_restricted','review_columns_exclusion_validated','interpretation_guardrail_created','09_handoff_questions_created','decision_summary_created','safe_unsafe_wording_created','open_risks_created','readme_created','note_md_updated','review_zip_created','notebook_saved_with_outputs']
st08a={str(p):stat(p) for p in f08}; details={n:'checked' for n in fc_names}; oks={n:True for n in fc_names}; oks.update({'repo_root_matches_expected':norm(actual)==norm(EXP),'source_file_exists':src.exists(),'source_file_inside_park_ingyeom':inside(src),'required_06_files_exist':all(p.exists() for p in f06),'required_05b_files_exist':all(p.exists() for p in f05),'required_07_files_exist':all(p.exists() for p in f07),'required_08_success_run_folder_exists':s08.exists(),'all_required_08_success_files_exist':all(p.exists() for p in f08),'notebook_inside_park_ingyeom':inside(NB),'output_folder_inside_park_ingyeom':inside(OUT),'zip_inside_park_ingyeom':inside(ZIPP),'no_files_written_outside_park_ingyeom':all(inside(p) for p in made+[OUT,ZIPP]),'no_source_csv_modified':src0==stat(src),'no_08_outputs_overwritten':st080==st08a,'no_files_deleted':all(p.exists() for p in f08+b08_before),'no_predictions_created':not vals['has_prediction_column'],'no_repurchase_score_created':not vals['has_repurchase_score_column'],'no_churn_risk_created':not vals['has_churn_risk_column'],'no_shap_performed':len(shap)==0,'no_statistical_tests_performed':len(pcols)==0,'no_p_values_created':len(pcols)==0,'negligible_promotion_feature_difference_documented':bc.get('moderate',0)==0 and bc.get('large',0)==0,'review_zip_created':True,'notebook_saved_with_outputs':True})
W(pd.DataFrame([{'check_name':n,'status':pf(oks[n]),'detail':details[n]} for n in fc_names]),'08b_final_checks.csv')
if ZIPP.exists(): ZIPP.unlink()
with zipfile.ZipFile(ZIPP,'w',zipfile.ZIP_DEFLATED) as z:
    if NB.exists(): z.write(NB,arcname=str(NB.relative_to(PARK)))
    for n in csvs:
        p=OUT/n
        if p.exists(): z.write(p,arcname=str(p.relative_to(PARK)))
    z.write(readme,arcname=str(readme.relative_to(PARK))); z.write(note,arcname=str(note.relative_to(PARK)))
print('valid 08 run folder:',s08.relative_to(PARK)); print('recomputed row counts:',{'primary_main_cohort':len(co),'nonpromotion':len(non),'promotion':len(pro)}); print('repurchase rates by promotion:',{'nonpromotion':nr,'promotion':pr,'difference_promo_minus_nonpromo':od}); print('max absolute SMD among conservative features:',mx); print('SMD bucket counts:',bc); print('decision on whether to rerun 08:', 'no' if accept else 'yes'); print('09 handoff summary: examine promotion x repurchase 2x2 cells with conservative safe features only.')
display(pd.DataFrame([{'repo_root':actual,'created_notebook':str(NB.relative_to(ROOT)),'output_folder':str(OUT.relative_to(ROOT)),'zip_path':str(ZIPP.relative_to(ROOT)),'accept_08_structure':'yes' if accept else 'no','rerun_08_full':'no' if accept else 'yes','patch_interpretation':'yes','proceed_to_09':'yes' if accept else 'no','max_abs_smd':mx,'smd_bucket_counts':bc}]))


valid 08 run folder: reports\eda\08_promotion_vs_nonpromotion_eda_260513\run_20260514_022322
recomputed row counts: {'primary_main_cohort': 23079, 'nonpromotion': 11175, 'promotion': 11904}
repurchase rates by promotion: {'nonpromotion': np.float64(0.7624161073825504), 'promotion': np.float64(0.6751512096774194), 'difference_promo_minus_nonpromo': np.float64(-0.08726489770513102)}
max absolute SMD among conservative features: 0.0264694645030762
SMD bucket counts: {'negligible': 22}
decision on whether to rerun 08: no
09 handoff summary: examine promotion x repurchase 2x2 cells with conservative safe features only.


,repo_root,created_notebook,output_folder,zip_path,accept_08_structure,rerun_08_full,patch_interpretation,proceed_to_09,max_abs_smd,smd_bucket_counts
0,C:/Code/ott-churn-prediction,park.ingyeom\notebook\08b_promotion_vs_nonprom...,park.ingyeom\reports\eda\08b_promotion_vs_nonp...,park.ingyeom\zip\08b_promotion_vs_nonpromotion...,yes,no,yes,yes,0.026469,{'negligible': 22}
